In [11]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
from transformers import TextStreamer
from tqdm.auto import tqdm

In [12]:
model_name = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=True)

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, #to make memory efficient
    trust_remote_code = False #to ensure security when loading code from the model repository 
)

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

In [16]:
# trying annotations
prompt = ('''
You are a legal document drafting assistant.

Your task: Generate a synthetic, full realistic financial agreement contract. 
These are material contracts involving money, obligations, or financial risk that public companies are required to report so investors can understand the company’s financial position and commitments. Includes loans, leases, and so on, everything that a public company is involved in when it comes to finance.
Requirements:
- Write entirely original text in the style of formal loan agreements. 
- Use only fictional company names, parties, places, dates, and amounts.
- Do NOT copy or paraphrase the example. Use it as stylistic reference only.
- Vary details and wording for diversity.

Example section (for reference, not for reuse/copying):

5. REPRESENTATIONS AND WARRANTIES
Borrower represents and warrants as follows:
5.1 Due Organization and Authorization. Borrower is duly organized, validly existing, and in good standing under the laws of its formation jurisdiction, and has all necessary authority to enter into and perform this Agreement...
5.2 Collateral. Borrower has good and marketable title to the Collateral, free and clear of all liens except those in favor of Lender...

Please write a new financial agreement following these instructions.
''')

inputs = tokenizer(prompt, return_tensors="pt")

generated_ids = model.generate(
    inputs["input_ids"],
    max_new_tokens=5000,
    do_sample=True,
)

decoded = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

# Write to file with annotation
with open("synthetic/generated_contract.txt", "a", encoding="utf-8") as f:
    f.write(decoded)

print(decoded)


You are a legal document drafting assistant.

Your task: Generate a synthetic, full realistic financial agreement contract. 
These are material contracts involving money, obligations, or financial risk that public companies are required to report so investors can understand the company’s financial position and commitments. Includes loans, leases, and so on, everything that a public company is involved in when it comes to finance.
Requirements:
- Write entirely original text in the style of formal loan agreements. 
- Use only fictional company names, parties, places, dates, and amounts.
- Do NOT copy or paraphrase the example. Use it as stylistic reference only.
- Vary details and wording for diversity.

Example section (for reference, not for reuse/copying):

5. REPRESENTATIONS AND WARRANTIES
Borrower represents and warrants as follows:
5.1 Due Organization and Authorization. Borrower is duly organized, validly existing, and in good standing under the laws of its formation jurisdictio

In [67]:
from faker import Faker

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

print(f"Agreement Date: {agreement_date}")
print(f"Lender: {lender}, Address: {lender_address}")
print(f"Borrower: {borrower}, Address: {borrower_address}")

Agreement Date: July 07, 2025
Lender: Kim, Perry and Smith, Address: 19368 Craig Alley, Staceyside, ND 82505
Borrower: Fisher and Sons, Address: 0097 Archer Course Suite 202, Daletown, GU 70217


In [17]:
from faker import Faker
from transformers import AutoTokenizer, AutoModelForCausalLM

fake = Faker()

lender = fake.company()
lender_address = fake.address().replace('\n', ', ')
borrower = fake.company()
borrower_address = fake.address().replace('\n', ', ')
agreement_date = fake.date_between(start_date='-10y', end_date='today').strftime("%B %d, %Y")

topic = (
    f"Loan Agreement\n"
    f"Lender: {lender}, {lender_address}\n"
    f"Borrower: {borrower}, {borrower_address}\n"
    f"Agreement Date: {agreement_date}\n"
)

sections = [
    "0. INTRODUCTION",
    "1. PARTIES",
    "2. DEFINITIONS",
    "3. LENDING DISCLOSURE",
    "4. LOAN TERMS",
    "5. REPAYMENT TERMS",
    "6. INTEREST RATES AND FEES",
    "7. COLLATERAL",
    "8. COVENANTS",
    "9. DEFAULT AND REMEDIES",
    "10. MISCELLANEOUS PROVISIONS"
]

output_file = "synthetic/generated_contract.txt"
contract_so_far = ""  # This accumulates the contract's text as context.

with open(output_file, "w", encoding="utf-8") as f:
    f.write("=== SYNTHETIC CONTRACT GENERATION ===\n\n")
    f.write(f"{topic}\n")
    f.write("="*40 + "\n\n")

for section in sections:
    # Build the prompt using prior generated contract content for context (trim if very long!)
    # For small models, keep context window in mind (e.g. use "contract_so_far[-1500:]" for long contracts)
    prompt = (
        "You are a legal document drafting assistant.\n"
        "You need to generate a full, realistic financial agreement contract section by section. Each section should be coherent with the previous sections and follow legal language and style.\n"
        f"Content:{topic}\nContract so far: {contract_so_far[-1500:]}\n"  # Only use last ~1500 chars for context to avoid overflow.
        f"Generate only the text for the following section: {section}\n"
        # "Continue the contract with this section, following legal language and style.\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(
        inputs["input_ids"], max_new_tokens=400, do_sample=True, pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Remove prompt echo (if present): get only the new text after the prompt
    if decoded.startswith(prompt):
        section_text = decoded[len(prompt):].strip()
    else:
        section_text = decoded.strip()

    # For the next iteration, accumulate the contract so far
    contract_so_far += f"\n\n{section}\n{section_text}"

    # Write to file with annotation
    with open(output_file, "a", encoding="utf-8") as f:
        f.write(f"\n\n--- Prompt for {section} ---\n")
        f.write(prompt.strip() + "\n")
        f.write(f"--- Output for {section} ---\n")
        f.write(section_text)
        f.write("\n" + "="*40 + "\n")

print(f"Done! See the generated contract in: {output_file}")

Done! See the generated contract in: synthetic/generated_contract.txt
